# Imputing missing values with FaissImputer 0.2

This notebook shows the corrected `fit`/`transform` workflow in FaissImputer 0.2.

- Complete rows supplied to `fit` become **donors**.
- For each incomplete row, neighbors are found using only that row's observed columns.
- Missing values are filled from the selected donors.
- A fully missing row falls back to statistics learned during `fit`.

The small values below are intentionally easy to verify by hand. This is a usage example, not a performance benchmark.

## 1. Import the package

When running this notebook from a cloned repository, install the project first with `pip install -e .` from the repository root.

In [1]:
from importlib.metadata import version

import numpy as np

from faiss_imputer import FaissImputer

print("faiss-imputer", version("faiss-imputer"))

faiss-imputer 0.2.0


## 2. Prepare fitting data and data with missing values

`X_train` has three complete rows, so all three can be donors. `X_missing` contains two partly observed rows and one fully missing row.

In [2]:
X_train = np.array(
    [
        [1.0, 10.0, 100.0],
        [2.0, 20.0, 200.0],
        [3.0, 30.0, 300.0],
    ],
    dtype=np.float32,
)

X_missing = np.array(
    [
        [1.5, np.nan, 150.0],
        [2.5, 25.0, np.nan],
        [np.nan, np.nan, np.nan],
    ],
    dtype=np.float32,
)

X_missing

array([[  1.5,   nan, 150. ],
       [  2.5,  25. ,   nan],
       [  nan,   nan,   nan]], dtype=float32)

## 3. Fit once, then transform

Here `n_neighbors=2` means that each partly observed row uses its two nearest complete donors. The default L2 metric is used, and the selected donor values are averaged.

In [3]:
original = X_missing.copy()

imputer = FaissImputer(n_neighbors=2, metric="l2", strategy="mean")
X_imputed = imputer.fit(X_train).transform(X_missing)

X_imputed

array([[  1.5,  15. , 150. ],
       [  2.5,  25. , 250. ],
       [  2. ,  20. , 200. ]], dtype=float32)

The results are:

- The first row has its first and third features observed. Its two closest donors supply `10` and `20`, so the missing middle value becomes `15`.
- The second row has its first and second features observed. Its two closest donors supply `200` and `300`, so the missing last value becomes `250`.
- The third row has nothing observed. It receives the column means learned from the fitting data: `2`, `20`, and `200`.

## 4. Verify the result

These assertions also confirm that `transform` did not modify the original input array.

In [4]:
expected = np.array(
    [
        [1.5, 15.0, 150.0],
        [2.5, 25.0, 250.0],
        [2.0, 20.0, 200.0],
    ],
    dtype=np.float32,
)

np.testing.assert_allclose(X_imputed, expected)
np.testing.assert_array_equal(X_missing, original)

print("All checks passed.")

All checks passed.


## Practical notes

- Input must be a two-dimensional numeric array.
- Fitting data must contain at least one complete row.
- `n_neighbors` cannot exceed the number of complete donor rows.
- With `metric="l2"`, scale features first when their numeric ranges differ greatly.
- `metric="ip"` uses raw inner product; it is not automatically cosine similarity.
- `strategy="median"` can be used instead of `"mean"`; it changes both donor aggregation and the fitted fallback statistics used for fully missing rows.